In [1]:
!ls /kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split/train

images	labels	labels.csv  masks


In [2]:
!pip install segmentation-models-pytorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.8 MB/s eta 0:00:00


In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm

mask_dir = "/kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split/train/masks"

all_values = set()

for file in tqdm(os.listdir(mask_dir)[:1000]):  # check first 1000 masks
    mask = cv2.imread(os.path.join(mask_dir, file), 0)
    all_values.update(np.unique(mask))

print("Unique labels across dataset sample:", sorted(all_values))

100%|██████████| 1000/1000 [00:18<00:00, 53.28it/s]

Unique labels across dataset sample: [np.uint8(0), np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5)]


In [4]:
import torch
print(torch.cuda.is_available())

True


In [5]:
# ================================
# FINAL SEGMENTATION PIPELINE
# HIGH-ACCURACY VERSION (PROJECT READY)
# ================================

import os
import torch
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ================================
# DATA PATHS
# ================================

DATA_ROOT = "/kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split"

TRAIN_IMG = f"{DATA_ROOT}/train/images"
TRAIN_MASK = f"{DATA_ROOT}/train/masks"

VAL_IMG = f"{DATA_ROOT}/val/images"
VAL_MASK = f"{DATA_ROOT}/val/masks"


# ================================
# IMAGE SETTINGS (IMPROVED)
# ================================

IMAGE_SIZE = 384
BATCH_SIZE = 6
NUM_CLASSES = 6
EPOCHS = 15


# ================================
# DATASET CLASS
# ================================

class FashionSegDataset(Dataset):

    def __init__(self, img_dir, mask_dir):

        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))


    def __len__(self):
        return len(self.images)


    def __getitem__(self, idx):

        img_name = self.images[idx]

        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        mask = cv2.imread(
            os.path.join(self.mask_dir, img_name.replace(".jpg", ".png")),
            0
        )

        mask = cv2.resize(
            mask,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        return torch.tensor(img), torch.tensor(mask).long()


# ================================
# LOAD DATA
# ================================

train_dataset = FashionSegDataset(TRAIN_IMG, TRAIN_MASK)
val_dataset = FashionSegDataset(VAL_IMG, VAL_MASK)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoader ready")


# ================================
# MODEL (UPGRADED)
# ================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
).to(device)


# ================================
# LOSS + OPTIMIZER
# ================================

loss_fn = smp.losses.DiceLoss(mode="multiclass")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# ================================
# METRICS
# ================================

def compute_iou(pred, mask):

    pred = pred.cpu().numpy()
    mask = mask.cpu().numpy()

    ious = []

    for cls in range(1, NUM_CLASSES):  # ignore background

        pred_inds = pred == cls
        target_inds = mask == cls

        intersection = (pred_inds & target_inds).sum()
        union = (pred_inds | target_inds).sum()

        if union == 0:
            continue

        ious.append(intersection / union)

    return np.mean(ious) if len(ious) > 0 else 0


def compute_dice(pred, mask):

    pred = pred.cpu().numpy().flatten()
    mask = mask.cpu().numpy().flatten()

    intersection = (pred == mask).sum()

    return (2 * intersection) / (len(pred) + len(mask) + 1e-6)


# ================================
# TRAIN LOOP
# ================================

best_miou = 0

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader):

        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = loss_fn(outputs, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print("Train Loss:", train_loss / len(train_loader))


    # ============================
    # VALIDATION
    # ============================

    model.eval()

    val_loss = 0
    miou_scores = []
    dice_scores = []

    with torch.no_grad():

        for imgs, masks in val_loader:

            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs = model(imgs)

            loss = loss_fn(outputs, masks)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            miou_scores.append(compute_iou(preds, masks))
            dice_scores.append(compute_dice(preds, masks))


    avg_val_loss = val_loss / len(val_loader)
    avg_miou = np.mean(miou_scores)
    avg_dice = np.mean(dice_scores)


    print("Validation Loss:", avg_val_loss)
    print("Mean IoU:", avg_miou)
    print("Dice Score:", avg_dice)


    # SAVE BEST MODEL (BY mIoU)

    if avg_miou > best_miou:

        best_miou = avg_miou

        torch.save(
            model.state_dict(),
            "/kaggle/working/unet_segmentation_best.pth"
        )

        print("Saved BEST segmentation model (highest mIoU)")


print("\nSEGMENTATION TRAINING COMPLETE")
print("Best mIoU achieved:", best_miou)

Using device: cuda
DataLoader ready


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


Epoch 1/15


100%|██████████| 2334/2334 [18:18<00:00,  2.12it/s]


...

Train Loss: 0.2384578052480428
Validation Loss: 0.2657145035862923
Mean IoU: 0.458192898517267
Dice Score: 0.8798632993339934
Saved BEST segmentation model (highest mIoU)

Epoch 14/15


100%|██████████| 2334/2334 [15:16<00:00,  2.55it/s]


Train Loss: 0.23213626271163232
Validation Loss: 0.26936170874163506
Mean IoU: 0.45263545469455246
Dice Score: 0.880816853840648

Epoch 15/15


100%|██████████| 2334/2334 [15:07<00:00,  2.57it/s]


Train Loss: 0.23025236757282222
Validation Loss: 0.25842712484300134
Mean IoU: 0.47059378096765353
Dice Score: 0.8840951538080942
Saved BEST segmentation model (highest mIoU)

SEGMENTATION TRAINING COMPLETE
Best mIoU achieved: 0.47059378096765353
